##**Mastering the Pinecone vector database.**

##**GOAL: save text sentence embeddings and perform semantic search in Pinecone using Python code.**

Create a Pinecone Account: Head to 'Pinecone.io' and sign up for a free account. After signing up, you’ll be given an API key.

 To start working with Pinecone,
install 'pinecone' library

In [1]:
!pip install -q pinecone

From 'pinecone' library import Pinecone, ServerlessSpec.

In [2]:
from pinecone import Pinecone, ServerlessSpec

import os
import time
from getpass import getpass

On the Pinecone main page, navigate to the "Get started" section via the left menu. Next, follow the "Quick Start for Developers" guide.
* Initialaze a client.
* Create an index.
* Upsert data.
* Perform a search.









When you perform "Upsert data", add a list of your own text strings.

When you perform "Search", use of your own text string.

In [3]:
# 1. client (key from env variable, otherwise type it in, so it's not saved in the notebook)
api_key = os.environ.get("PINECONE_API_KEY") or getpass("Pinecone API key: ")
pc = Pinecone(api_key=api_key)

# 2. index, multilingual-e5-large gives vectors of size 1024
index_name = "day9-notes"
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1024,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

while not pc.describe_index(index_name).status["ready"]:
    time.sleep(1)

index = pc.Index(index_name)

# 3. upsert my own sentences, a few of them about sleep for the search below
my_texts = [
    "Drinking enough water during the day helps you stay focused and feel less tired.",
    "Cats sleep for around twelve to sixteen hours every day.",
    "The Eiffel Tower was built for the World's Fair in Paris in 1889.",
    "Regular exercise, even a short daily walk, is good for your heart.",
    "Python is a popular programming language for data science and machine learning.",
    "Tomatoes grow best in warm weather with a lot of sunlight.",
    "The Great Wall of China is thousands of kilometres long.",
    "Reading a book before bed instead of using your phone can help you fall asleep faster.",
    "Going to bed and waking up at the same time every day keeps your sleep rhythm steady.",
    "A cool, dark and quiet bedroom makes it easier to sleep through the night.",
    "Drinking coffee late in the afternoon can keep you awake for hours at night.",
]

passages = pc.inference.embed(
    model="multilingual-e5-large",
    inputs=my_texts,
    parameters={"input_type": "passage", "truncate": "END"},
)

vectors = []
for i, (text, emb) in enumerate(zip(my_texts, passages)):
    vectors.append({"id": f"note{i}", "values": emb["values"], "metadata": {"text": text}})

index.upsert(vectors=vectors, namespace="notes")
time.sleep(10)  # new vectors need a few seconds before they show up in search
print(index.describe_index_stats())

# 4. search with my own question
question = "How can I sleep better at night?"
q_emb = pc.inference.embed(
    model="multilingual-e5-large",
    inputs=[question],
    parameters={"input_type": "query"},
)

results = index.query(
    namespace="notes",
    vector=q_emb[0]["values"],
    top_k=3,
    include_metadata=True,
)

DescribeIndexStatsResponse(dimension=1024, total_vector_count=11, metric='cosine', namespaces=1)


Print search results.

In [4]:
print("question:", question)
for match in results["matches"]:
    print(round(match["score"], 3), "|", match["metadata"]["text"])

# free plan has few indexes, uncomment to remove it after
# pc.delete_index(index_name)

question: How can I sleep better at night?
0.847 | A cool, dark and quiet bedroom makes it easier to sleep through the night.
0.846 | Going to bed and waking up at the same time every day keeps your sleep rhythm steady.
0.84 | Reading a book before bed instead of using your phone can help you fall asleep faster.


### What I noticed

For "How can I sleep better at night?" all top 3 results are sleep tips (dark bedroom, fixed bedtime, reading before bed), so the search works even though the words are different. The e5 scores are very close to each other (0.84 to 0.85), so the order of the results says more than the score itself. The API key is read from an environment variable, so it is not saved in the notebook.